# Advanced Covenant Modeling Demo

This notebook mirrors the advanced covenant modeling script. It demonstrates:

- Maintenance vs. Incurrence scopes on covenants
- Springing conditions that activate leverage tests only when revolver utilization is high
- Basket utilization tracking with explicit headroom analytics



In [1]:
from finstack import (
    Covenant,
    CovenantForecastConfig,
    CovenantScope,
    CovenantSpec,
    CovenantType,
    SpringingCondition,
    forecast_covenant,
)
from finstack.core.dates.periods import build_periods
from finstack.statements.evaluator import Evaluator
from finstack.statements.types import (
    AmountOrScalar,
    FinancialModelSpec,
    ForecastSpec,
    NodeSpec,
    NodeType,
)



In [2]:
def build_extended_model():
    plan = build_periods("2024Q1..2026Q4", None)
    periods = plan.periods
    model = FinancialModelSpec("advanced_covenants_demo", periods)

    ebitda = (
        NodeSpec("ebitda", NodeType.MIXED)
        .with_values([(periods[0].id, AmountOrScalar.scalar(55_000.0))])
        .with_forecast(ForecastSpec.growth(0.04))
    )

    debt_levels = [
        (
            periods[i].id,
            AmountOrScalar.scalar(250_000.0 if i < 6 else 225_000.0),
        )
        for i in range(len(periods))
    ]
    debt_total = NodeSpec("debt_total", NodeType.MIXED).with_values(debt_levels)

    total_leverage = NodeSpec("total_leverage", NodeType.CALCULATED).with_formula(
        "debt_total / ebitda"
    )

    utilization_values = []
    for idx, period in enumerate(periods):
        utilization = 0.25 if idx < 4 else (0.42 if idx < 8 else 0.58)
        utilization_values.append((period.id, AmountOrScalar.scalar(utilization)))
    rcf_utilization = NodeSpec("rcf_utilization", NodeType.MIXED).with_values(
        utilization_values
    )

    basket_usage = []
    for idx, period in enumerate(periods):
        drawn = 60.0 + idx * 5.0
        basket_usage.append((period.id, AmountOrScalar.scalar(drawn)))
    general_debt = NodeSpec("general_debt_basket", NodeType.MIXED).with_values(
        basket_usage
    )

    for node in (ebitda, debt_total, total_leverage, rcf_utilization, general_debt):
        model.add_node(node)

    return model, periods


def describe_forecast(label, forecast, warn_threshold=0.1):
    print(f"\n== {label} ==")
    print(f"First breach date: {forecast.first_breach_date}")
    print(
        f"Minimum headroom: {forecast.min_headroom_value:.1%} on {forecast.min_headroom_date}"
    )

    warn_indices = [
        idx for idx, headroom in enumerate(forecast.headroom) if headroom < warn_threshold
    ]
    if warn_indices:
        warn_dates = [forecast.test_dates[idx] for idx in warn_indices]
        print(f"Warning: headroom < {warn_threshold:.0%} on {warn_dates}")
    else:
        print(f"No periods under {warn_threshold:.0%} headroom.")

    print("\n-- Explain --")
    print(forecast.explain())



In [ ]:
model, periods = build_extended_model()
evaluator = Evaluator.new()
results = evaluator.evaluate(model)
window = periods[4:12]
window_ids = [p.id for p in window]
cfg = CovenantForecastConfig(stochastic=False, num_paths=0, volatility=None, seed=None, antithetic=False)

springing = SpringingCondition("rcf_utilization", "minimum", 0.35)
leverage_cov = (
    Covenant(CovenantType.max_total_leverage(4.5))
    .with_scope(CovenantScope.maintenance())
    .with_springing_condition(springing)
)
leverage_spec = CovenantSpec.with_metric(leverage_cov, "total_leverage")
leverage_forecast = forecast_covenant(leverage_spec, model, results, window_ids, cfg)
describe_forecast("Springing Maintenance Covenant", leverage_forecast)

basket_cov = Covenant(CovenantType.basket("general_debt_basket", 120.0)).with_scope(
    CovenantScope.incurrence()
)
basket_spec = CovenantSpec.with_metric(basket_cov, "general_debt_basket")
basket_forecast = forecast_covenant(basket_spec, model, results, window_ids, cfg)
describe_forecast("General Debt Basket", basket_forecast, warn_threshold=0.2)



TypeError: CovenantForecastConfig.__new__() missing 5 required positional arguments: 'stochastic', 'num_paths', 'volatility', 'seed', and 'antithetic'

In [ ]:
# Quickly inspect the headroom vectors
print("Leverage headroom (first 5):", [f"{h:.1%}" for h in leverage_forecast.headroom[:5]])
print("Basket headroom (first 5):  ", [f"{h:.1%}" for h in basket_forecast.headroom[:5]])

